In [1]:

import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset




class SimpleModel(nn.Module):

    def __init__(self):

        super(SimpleModel, self).__init__()

        self.fc = nn.Linear(5, 2)

    def forward(self, x):

        return self.fc(x)






def create_client_data(samples):

    X = torch.randn(samples, 5)

    y = torch.randint(0, 2, (samples,))

    return TensorDataset(X, y)





def local_training(model, dataset):

    loader = DataLoader(dataset, batch_size=8, shuffle=True)

    loss_function = nn.CrossEntropyLoss()

    optimizer = optim.SGD(model.parameters(), lr=0.01)

    model.train()

    for epoch in range(2):

        for X, y in loader:

            optimizer.zero_grad()

            output = model(X)

            loss = loss_function(output, y)

            loss.backward()

            optimizer.step()

    return model.state_dict()




def weighted_fedavg(global_model, client_models, client_sizes):

    total_data = sum(client_sizes)

    global_state = copy.deepcopy(global_model.state_dict())

    for key in global_state.keys():

        global_state[key] = torch.zeros_like(global_state[key])

        for client_state, size in zip(client_models, client_sizes):

            weight = size / total_data

            global_state[key] += client_state[key] * weight

    global_model.load_state_dict(global_state)

    return global_model


global_model = SimpleModel()

client_sizes = [50, 100, 150]

client_datasets = [
    create_client_data(50),
    create_client_data(100),
    create_client_data(150)
]

rounds = 3

for round_num in range(rounds):

    print(f"\nFederated Round {round_num + 1}")

    client_models = []


    for i in range(3):

        print(f"Client {i+1} Training...")

        local_model = copy.deepcopy(global_model)

        updated_model = local_training(
            local_model,
            client_datasets[i]
        )

        client_models.append(updated_model)





    global_model = weighted_fedavg(
        global_model,
        client_models,
        client_sizes
    )

    print("Global Model Updated")

    print("Global Model Distributed to Clients")



print("\nFederated Learning Completed Successfully")

ModuleNotFoundError: No module named 'torch'